# Rossmann Store Sales: Modeling

includes the features and time split to make sure every model trains on the same data and nothing leaks and all four models are evaluated on the same open july days.

predictions and metrics are saved in the end for results

## Load the cleaned data

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
np.random.seed(42)
pd.set_option("display.max_columns",None)#no hidden cols

#PROC = Path.cwd().parent
PROC = Path.cwd().parent /"data"/"processed"
#StateHoliday as text and Date as real dates so the types come back the same as the eda
clean = pd.read_csv(PROC / "rossmann_clean.csv", parse_dates=['Date'], dtype={"StateHoliday" : str})
print(clean.shape)
clean.head()

(1050330, 13)


,Store,Date,Sales,Open,Promo,StateHoliday,SchoolHoliday,was_present,DayOfWeek,StoreType,Assortment,CompetitionDistance,Promo2
0,1,2013-01-01,0.0,0,0,a,1,True,2,c,a,1270.0,0
1,1,2013-01-02,5530.0,1,0,0,1,True,3,c,a,1270.0,0
2,1,2013-01-03,4327.0,1,0,0,1,True,4,c,a,1270.0,0
3,1,2013-01-04,4486.0,1,0,0,1,True,5,c,a,1270.0,0
4,1,2013-01-05,4997.0,1,0,0,1,True,6,c,a,1270.0,0


## Feature engineering

new sales-history features for each store

each store already has a full daily calender now after eda, so no gap can leak now (so for example a lag 7 reliably means exactly 7 days ago now)

I chose the window sizes based on the weekly pattern found during eda

In [2]:
clean = clean.sort_values(["Store","Date"]).reset_index(drop=True)
g = clean.groupby("Store")["Sales"]
clean["lag1"]=g.shift(1)#yesterday's sales of same store
clean["lag7"]=g.shift(7)#last week (same day of week) sales of same stroe (this was a strong one)
clean["lag14"]=g.shift(14)# 2 weeks ago sales of same store (same day of week)

In [3]:
# I made sure rolling mean first does shift 1 to make sure each today is never inside its own avrg
clean["rollmean7"] = g.transform(lambda s: s.shift(1).rolling(7).mean()) # avrg of last 7 days
clean["rollmean14"] = g.transform(lambda s: s.shift(1).rolling(14).mean())#avrg of last 14 days

In [4]:
#two more extra to try
clean["rollmean28"] = g.transform(lambda s: s.shift(1).rolling(28).mean())#avrg of last 28 days
clean["rollstd7"] = g.transform(lambda s: s.shift(1).rolling(7).std())#how much sales changed in last 7 days

In [5]:
clean["month"] = clean["Date"].dt.month #making month col using date
clean["weekofyear"] = clean["Date"].dt.isocalendar().week.astype(int)# making week number col (was really helpful for december peaks)
#clean["weekofyear"] = clean["Date"].dt.isocalendar().week
#print(clean[["Date","month","weekofyear"]].tail())

In [6]:
clean[clean.Store==1][["Date","Sales","lag7"]].head(9)

,Date,Sales,lag7
0,2013-01-01,0.0,NaN
1,2013-01-02,5530.0,NaN
2,2013-01-03,4327.0,NaN
3,2013-01-04,4486.0,NaN
4,2013-01-05,4997.0,NaN
5,2013-01-06,0.0,NaN
6,2013-01-07,7176.0,NaN
7,2013-01-08,5580.0,0.0
8,2013-01-09,5471.0,5530.0
